In [ ]:
import importlib.util
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import zipfile


COMPETITION = "store-sales-time-series-forecasting"
FILES = (
    "train.csv",
    "test.csv",
    "stores.csv",
    "oil.csv",
    "holidays_events.csv",
    "transactions.csv",
    "sample_submission.csv",
)


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        if (directory / "notebooks" / "data_installation.ipynb").is_file():
            return directory
    raise RuntimeError("Start the notebook kernel from the yumeko project or its notebooks folder.")


def install(destination: Path, force: bool = False) -> None:
    destination = destination.resolve()
    needed = [
        name for name in FILES
        if force or not (destination / name).is_file()
        or (destination / name).stat().st_size == 0
    ]
    if not needed:
        print(f"All {len(FILES)} CSV files are already present in {destination}.")
        return

    if importlib.util.find_spec("kaggle") is None:
        raise RuntimeError(
            "Kaggle is not installed. Run: python -m pip install -r requirements.txt"
        )

    with tempfile.TemporaryDirectory(prefix="yumeko-data-") as temporary:
        try:
            subprocess.run(
                [
                    sys.executable, "-m", "kaggle", "competitions", "download",
                    "-c", COMPETITION, "-p", temporary,
                ],
                check=True,
                capture_output=True,
                text=True,
                errors="replace",
            )
        except subprocess.CalledProcessError as error:
            details = (error.stderr or error.stdout or "").strip()
            raise RuntimeError(
                "Kaggle download failed. Check your Kaggle login and confirm "
                "that you accepted the competition rules.\n" + details
            ) from None
        archive_path = Path(temporary) / f"{COMPETITION}.zip"
        with zipfile.ZipFile(archive_path) as archive:
            # Only copy the expected filenames; never extract arbitrary ZIP paths.
            for name in needed:
                info = archive.getinfo(name)
                if info.is_dir() or info.file_size == 0:
                    raise RuntimeError(f"Archive contains an empty or invalid {name}.")
            staged = Path(temporary) / "extracted"
            staged.mkdir()
            for name in needed:
                with archive.open(name) as source, (staged / name).open("wb") as target:
                    shutil.copyfileobj(source, target)

        destination.mkdir(parents=True, exist_ok=True)
        for name in needed:
            shutil.copyfile(staged / name, destination / name)
            print(f"Installed {name}")
    print(f"Data ready in {destination}")


install(find_project_root() / "data" / "raw")